Accession Range Expander
========================
Utility for expanding accession number ranges and updating ground truth files.

Takes range boundaries (e.g., MH548440, MH548519) and generates the complete
list of accessions. Can also update ground truth JSON files with expanded ranges.

AI6129 Project
Date: January 2026

In [10]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Set
from dataclasses import dataclass
from datetime import datetime
from copy import deepcopy
from google.colab import drive

In [11]:
# ============================================================================
# Configuration
# ============================================================================

drive.mount('/content/drive')

# Declare configuration variables at the start
XML_DIRECTORY = "/content/drive/MyDrive/AI6129/xml"
OUTPUT_DIRECTORY = "/content/drive/MyDrive/AI6129/utility_output"
GT_DIRECTORY = "/content/drive/MyDrive/AI6129/ground_truth"
LOG_LEVEL = logging.INFO

# Backup configuration
CREATE_BACKUPS = True
BACKUP_SUFFIX = '.backup'


# ============================================================================
# Data Classes
# ============================================================================

@dataclass
class ExpansionResult:
    """Result of expanding an accession range."""
    start_accession: str
    end_accession: str
    expanded_list: List[str]
    count: int
    success: bool
    error_message: str = ""


@dataclass
class GTUpdateResult:
    """Result of updating a ground truth file."""
    pmcid: str
    filepath: str
    original_count: int
    new_count: int
    added_accessions: List[str]
    success: bool
    error_message: str = ""


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ============================================================================
# Core Expansion Functions
# ============================================================================

def setup_logging(log_file: Optional[str] = None) -> logging.Logger:
    """
    Configure logging for the range expander.

    Args:
        log_file: Optional path to log file

    Returns:
        Configured logger instance
    """
    logger = logging.getLogger('RangeExpander')
    logger.setLevel(LOG_LEVEL)

    # Clear existing handlers
    logger.handlers = []

    formatter = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # File handler (optional)
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    return logger


def parse_accession(accession: str) -> Tuple[Optional[str], Optional[int], Optional[int]]:
    """
    Parse an accession number into components.

    Args:
        accession: Accession string (e.g., 'MH548440' or 'MH548440.1')

    Returns:
        Tuple of (prefix, number, digit_width) or (None, None, None) if invalid
    """
    # Remove version suffix if present
    accession_clean = accession.split('.')[0].strip().upper()

    # Match pattern: 2 letters + digits
    match = re.match(r'^([A-Z]{2})(\d+)$', accession_clean)

    if not match:
        return None, None, None

    prefix = match.group(1)
    num_str = match.group(2)
    number = int(num_str)
    digit_width = len(num_str)

    return prefix, number, digit_width


def expand_range(start_accession: str, end_accession: str) -> ExpansionResult:
    """
    Expand an accession range into a complete list.

    Args:
        start_accession: Starting accession (e.g., 'MH548440')
        end_accession: Ending accession (e.g., 'MH548519')

    Returns:
        ExpansionResult with expanded list or error
    """
    # Parse both accessions
    start_prefix, start_num, start_width = parse_accession(start_accession)
    end_prefix, end_num, end_width = parse_accession(end_accession)

    # Validate parsing
    if start_prefix is None:
        return ExpansionResult(
            start_accession=start_accession,
            end_accession=end_accession,
            expanded_list=[],
            count=0,
            success=False,
            error_message=f"Invalid start accession format: {start_accession}"
        )

    if end_prefix is None:
        return ExpansionResult(
            start_accession=start_accession,
            end_accession=end_accession,
            expanded_list=[],
            count=0,
            success=False,
            error_message=f"Invalid end accession format: {end_accession}"
        )

    # Validate prefix match
    if start_prefix != end_prefix:
        return ExpansionResult(
            start_accession=start_accession,
            end_accession=end_accession,
            expanded_list=[],
            count=0,
            success=False,
            error_message=f"Prefix mismatch: {start_prefix} vs {end_prefix}"
        )

    # Validate range direction
    if end_num < start_num:
        return ExpansionResult(
            start_accession=start_accession,
            end_accession=end_accession,
            expanded_list=[],
            count=0,
            success=False,
            error_message=f"End number ({end_num}) is less than start ({start_num})"
        )

    # Use the larger digit width to preserve leading zeros
    digit_width = max(start_width, end_width)

    # Generate expanded list
    expanded = []
    for num in range(start_num, end_num + 1):
        accession = f"{start_prefix}{num:0{digit_width}d}"
        expanded.append(accession)

    return ExpansionResult(
        start_accession=start_accession.upper(),
        end_accession=end_accession.upper(),
        expanded_list=expanded,
        count=len(expanded),
        success=True
    )


def expand_multiple_ranges(ranges: List[Tuple[str, str]]) -> List[ExpansionResult]:
    """
    Expand multiple accession ranges.

    Args:
        ranges: List of (start, end) tuples

    Returns:
        List of ExpansionResult objects
    """
    results = []

    for start, end in ranges:
        result = expand_range(start, end)
        results.append(result)

    return results


def merge_accession_lists(existing: List[str],
                          new_accessions: List[str]) -> Tuple[List[str], List[str], bool]:
    """
    Merge new accessions into existing list, avoiding duplicates.

    Args:
        existing: Existing accession list (may have [GenBank] suffix)
        new_accessions: New accessions to add (clean, without suffix)

    Returns:
        Tuple of (merged_list_clean, added_accessions, had_genbank_suffix)
    """
    # Check if existing accessions have [GenBank] suffix
    has_genbank_suffix = False
    if existing and len(existing) > 0:
        has_genbank_suffix = '[GenBank]' in existing[0]

    # Normalise existing to uppercase without suffix for comparison
    existing_clean = set()
    for acc in existing:
        clean = strip_genbank_suffix(acc).upper().split('.')[0]
        existing_clean.add(clean)

    # Find truly new accessions
    added = []
    for acc in new_accessions:
        acc_normalised = acc.upper().split('.')[0]
        if acc_normalised not in existing_clean:
            added.append(acc_normalised)
            existing_clean.add(acc_normalised)

    # Combine and sort
    merged = sorted(existing_clean)

    return merged, added, has_genbank_suffix

In [13]:
# ============================================================================
# Ground Truth Update Functions
# ============================================================================

def load_ground_truth(filepath: str) -> Tuple[Optional[Dict], str]:
    """
    Load a ground truth JSON file.

    Args:
        filepath: Path to JSON file

    Returns:
        Tuple of (data_dict, error_message)
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data, ""
    except FileNotFoundError:
        return None, f"File not found: {filepath}"
    except json.JSONDecodeError as e:
        return None, f"JSON decode error: {e}"
    except Exception as e:
        return None, f"Error loading file: {e}"


def save_ground_truth(filepath: str, data: Dict,
                      create_backup: bool = True) -> Tuple[bool, str]:
    """
    Save ground truth JSON file with optional backup.

    Args:
        filepath: Path to JSON file
        data: Data dictionary to save
        create_backup: Whether to create backup of original

    Returns:
        Tuple of (success, error_message)
    """
    try:
        # Create backup if requested
        if create_backup and os.path.exists(filepath):
            backup_path = filepath + BACKUP_SUFFIX
            with open(filepath, 'r', encoding='utf-8') as f:
                original = f.read()
            with open(backup_path, 'w', encoding='utf-8') as f:
                f.write(original)

        # Write updated data
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        return True, ""

    except Exception as e:
        return False, f"Error saving file: {e}"


def extract_accessions_from_gt(data: Dict) -> Tuple[List[str], str]:
    """
    Extract accession numbers from ground truth data.

    Handles multiple field formats:
    - "merge_accession_number": list format with [GenBank] suffix
    - "Accession Numbers": list format
    - "Accession Numbers_merge": string format

    Args:
        data: Ground truth dictionary

    Returns:
        Tuple of (list of accession numbers, field_name used)
    """
    accessions = []
    field_name = None

    # Try merge_accession_number first (primary format)
    if 'merge_accession_number' in data:
        field_name = 'merge_accession_number'
        field_value = data['merge_accession_number']
        if isinstance(field_value, list):
            accessions = field_value
        elif isinstance(field_value, str):
            accessions = [a.strip() for a in field_value.split(',') if a.strip()]

    # Try Accession Numbers as fallback
    elif 'Accession Numbers' in data:
        field_name = 'Accession Numbers'
        field_value = data['Accession Numbers']
        if isinstance(field_value, list):
            accessions = field_value
        elif isinstance(field_value, str):
            accessions = [a.strip() for a in field_value.split(',') if a.strip()]

    # Try Accession Numbers_merge as final fallback
    elif 'Accession Numbers_merge' in data:
        field_name = 'Accession Numbers_merge'
        field_value = data['Accession Numbers_merge']
        if isinstance(field_value, str):
            accessions = [a.strip() for a in field_value.split(',') if a.strip()]
        elif isinstance(field_value, list):
            accessions = field_value

    return accessions, field_name


def strip_genbank_suffix(accession: str) -> str:
    """
    Remove [GenBank] suffix from accession if present.

    Args:
        accession: Accession string, possibly with suffix

    Returns:
        Clean accession number
    """
    # Remove [GenBank] or similar suffixes
    clean = re.sub(r'\[.*?\]', '', accession).strip()
    return clean


def add_genbank_suffix(accession: str) -> str:
    """
    Add [GenBank] suffix to accession.

    Args:
        accession: Clean accession number

    Returns:
        Accession with [GenBank] suffix
    """
    return f"{accession}[GenBank]"


def update_accessions_in_gt(data: Dict, new_accessions: List[str],
                            field_name: str, use_genbank_suffix: bool) -> Dict:
    """
    Update accession numbers in ground truth data.

    Args:
        data: Ground truth dictionary
        new_accessions: New accession list to set (clean, without suffix)
        field_name: Name of the field to update
        use_genbank_suffix: Whether to add [GenBank] suffix to each accession

    Returns:
        Updated data dictionary
    """
    updated = deepcopy(data)

    # Format accessions with suffix if needed
    if use_genbank_suffix:
        formatted = [add_genbank_suffix(acc) for acc in new_accessions]
    else:
        formatted = new_accessions

    # Update the specific field that was found
    if field_name and field_name in updated:
        updated[field_name] = formatted
    elif field_name:
        # Field name provided but not in data - create it
        updated[field_name] = formatted
    else:
        # No field name - default to merge_accession_number
        updated['merge_accession_number'] = formatted

    return updated

In [14]:
def update_gt_file_with_ranges(gt_filepath: str,
                                ranges: List[Tuple[str, str]],
                                logger: logging.Logger) -> GTUpdateResult:
    """
    Update a single ground truth file with expanded ranges.

    Args:
        gt_filepath: Path to ground truth JSON file
        ranges: List of (start, end) range tuples to expand
        logger: Logger instance

    Returns:
        GTUpdateResult with outcome
    """
    # Extract PMCID from filename
    filename = os.path.basename(gt_filepath)
    pmcid_match = re.search(r'PMC\d+', filename)
    pmcid = pmcid_match.group(0) if pmcid_match else filename

    # Load existing ground truth
    data, error = load_ground_truth(gt_filepath)
    if data is None:
        return GTUpdateResult(
            pmcid=pmcid,
            filepath=gt_filepath,
            original_count=0,
            new_count=0,
            added_accessions=[],
            success=False,
            error_message=error
        )

    # Get existing accessions and field name
    existing, field_name = extract_accessions_from_gt(data)
    original_count = len(existing)

    if field_name is None:
        field_name = 'merge_accession_number'  # Default field name

    # Expand all ranges
    all_expanded = []
    for start, end in ranges:
        result = expand_range(start, end)
        if result.success:
            all_expanded.extend(result.expanded_list)
        else:
            logger.warning(f"{pmcid}: Range expansion failed for {start}-{end}: {result.error_message}")

    # Merge with existing (returns clean accessions without suffix)
    merged, added, has_genbank_suffix = merge_accession_lists(existing, all_expanded)

    # Update data with correct field and format
    updated_data = update_accessions_in_gt(data, merged, field_name, has_genbank_suffix)

    # Save updated file
    success, save_error = save_ground_truth(gt_filepath, updated_data, CREATE_BACKUPS)

    if not success:
        return GTUpdateResult(
            pmcid=pmcid,
            filepath=gt_filepath,
            original_count=original_count,
            new_count=original_count,
            added_accessions=[],
            success=False,
            error_message=save_error
        )

    logger.info(f"{pmcid}: Updated GT from {original_count} to {len(merged)} accessions (+{len(added)})")

    return GTUpdateResult(
        pmcid=pmcid,
        filepath=gt_filepath,
        original_count=original_count,
        new_count=len(merged),
        added_accessions=added,
        success=True
    )


def batch_update_gt_from_detection_report(detection_report_path: str,
                                           gt_directory: str,
                                           logger: logging.Logger) -> List[GTUpdateResult]:
    """
    Batch update ground truth files based on range detection report.

    Args:
        detection_report_path: Path to range_detection_detailed.json
        gt_directory: Directory containing ground truth JSON files
        logger: Logger instance

    Returns:
        List of GTUpdateResult objects
    """
    # Load detection report
    with open(detection_report_path, 'r', encoding='utf-8') as f:
        detection_data = json.load(f)

    results = []

    for doc in detection_data:
        pmcid = doc['pmcid']

        # Find corresponding GT file
        gt_pattern = f"*{pmcid}*.json"
        gt_files = list(Path(gt_directory).glob(gt_pattern))

        if not gt_files:
            logger.warning(f"{pmcid}: No matching GT file found in {gt_directory}")
            results.append(GTUpdateResult(
                pmcid=pmcid,
                filepath="",
                original_count=0,
                new_count=0,
                added_accessions=[],
                success=False,
                error_message="No matching GT file found"
            ))
            continue

        gt_filepath = str(gt_files[0])

        # Extract ranges from detection report
        ranges = [
            (r['start_accession'], r['end_accession'])
            for r in doc['ranges']
        ]

        # Update GT file
        result = update_gt_file_with_ranges(gt_filepath, ranges, logger)
        results.append(result)

    return results

In [15]:
# ============================================================================
# Reporting Functions
# ============================================================================

def generate_update_report(results: List[GTUpdateResult],
                           output_path: str) -> Dict:
    """
    Generate report of GT update results.

    Args:
        results: List of update results
        output_path: Path for output file

    Returns:
        Summary statistics
    """
    successful = [r for r in results if r.success]
    failed = [r for r in results if not r.success]

    total_original = sum(r.original_count for r in successful)
    total_new = sum(r.new_count for r in successful)
    total_added = sum(len(r.added_accessions) for r in successful)

    summary = {
        'update_timestamp': datetime.now().isoformat(),
        'total_files_processed': len(results),
        'successful_updates': len(successful),
        'failed_updates': len(failed),
        'total_original_accessions': total_original,
        'total_new_accessions': total_new,
        'total_accessions_added': total_added,
        'details': {
            'successful': [
                {
                    'pmcid': r.pmcid,
                    'original': r.original_count,
                    'new': r.new_count,
                    'added': len(r.added_accessions)
                }
                for r in successful
            ],
            'failed': [
                {
                    'pmcid': r.pmcid,
                    'error': r.error_message
                }
                for r in failed
            ]
        }
    }

    # Write report
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    return summary


def print_update_summary(summary: Dict) -> None:
    """
    Print update summary to console.

    Args:
        summary: Summary statistics dictionary
    """
    print("\n" + "=" * 70)
    print("GROUND TRUTH UPDATE SUMMARY")
    print("=" * 70)
    print(f"Update timestamp:            {summary['update_timestamp']}")
    print(f"Files processed:             {summary['total_files_processed']}")
    print(f"Successful updates:          {summary['successful_updates']}")
    print(f"Failed updates:              {summary['failed_updates']}")
    print("-" * 70)
    print(f"Original accessions:         {summary['total_original_accessions']}")
    print(f"New accessions:              {summary['total_new_accessions']}")
    print(f"Accessions added:            {summary['total_accessions_added']}")
    print("=" * 70 + "\n")



In [16]:
# ============================================================================
# Command-Line Interface Functions
# ============================================================================

def expand_single_range_cli(start: str, end: str) -> None:
    """
    CLI function to expand a single range and print results.

    Args:
        start: Start accession
        end: End accession
    """
    result = expand_range(start, end)

    if result.success:
        print(f"\nExpanded range: {result.start_accession} to {result.end_accession}")
        print(f"Total accessions: {result.count}")
        print("\nAccession list:")

        # Print in columns for readability
        cols = 5
        for i in range(0, len(result.expanded_list), cols):
            row = result.expanded_list[i:i+cols]
            print("  " + "  ".join(row))
    else:
        print(f"\nError: {result.error_message}")


def update_from_detection_cli(detection_report: str, gt_directory: str,
                               output_report: str) -> None:
    """
    CLI function to batch update GT files from detection report.

    Args:
        detection_report: Path to detection report JSON
        gt_directory: Directory containing GT files
        output_report: Path for update report output
    """
    logger = setup_logging()

    logger.info("Starting batch GT update from detection report...")

    results = batch_update_gt_from_detection_report(
        detection_report, gt_directory, logger
    )

    summary = generate_update_report(results, output_report)
    print_update_summary(summary)

    logger.info(f"Update report written to: {output_report}")

In [17]:
# ============================================================================
# Main Entry Point
# ============================================================================

def main():
    """Main entry point with argument parsing."""
    import argparse

    parser = argparse.ArgumentParser(
        description='Expand accession number ranges and update ground truth files'
    )

    subparsers = parser.add_subparsers(dest='command', help='Command to run')

    # Subcommand: expand
    expand_parser = subparsers.add_parser(
        'expand',
        help='Expand a single accession range'
    )
    expand_parser.add_argument('start', help='Start accession (e.g., MH548440)')
    expand_parser.add_argument('end', help='End accession (e.g., MH548519)')

    # Subcommand: update
    update_parser = subparsers.add_parser(
        'update',
        help='Batch update GT files from detection report'
    )
    update_parser.add_argument(
        'detection_report',
        help='Path to range_detection_detailed.json'
    )
    update_parser.add_argument(
        'gt_directory',
        help='Directory containing ground truth JSON files'
    )
    update_parser.add_argument(
        'output_report',
        help='Path for update report output'
    )

    # Subcommand: update-single
    single_parser = subparsers.add_parser(
        'update-single',
        help='Update a single GT file with specific ranges'
    )
    single_parser.add_argument(
        'gt_file',
        help='Path to ground truth JSON file'
    )
    single_parser.add_argument(
        'ranges',
        nargs='+',
        help='Ranges in format START:END (e.g., MH548440:MH548519)'
    )

    args = parser.parse_args()

    if args.command == 'expand':
        expand_single_range_cli(args.start, args.end)

    elif args.command == 'update':
        update_from_detection_cli(
            args.detection_report,
            args.gt_directory,
            args.output_report
        )

    elif args.command == 'update-single':
        logger = setup_logging()
        ranges = []
        for r in args.ranges:
            parts = r.split(':')
            if len(parts) == 2:
                ranges.append((parts[0], parts[1]))
            else:
                print(f"Invalid range format: {r} (expected START:END)")
                return

        result = update_gt_file_with_ranges(args.gt_file, ranges, logger)

        if result.success:
            print(f"\nSuccessfully updated {result.pmcid}")
            print(f"  Original: {result.original_count} accessions")
            print(f"  New: {result.new_count} accessions")
            print(f"  Added: {len(result.added_accessions)} accessions")
        else:
            print(f"\nFailed to update: {result.error_message}")

    else:
        parser.print_help()



In [18]:
DETECTION_REPORT = f'{OUTPUT_DIRECTORY}/range_detection_detailed.json'
UPDATE_REPORT = f'{OUTPUT_DIRECTORY}/gt_update_report.json'

# Setup logger
logger = setup_logging(log_file=f'{OUTPUT_DIRECTORY}/accession_expand.log')

# Run batch update
results = batch_update_gt_from_detection_report(
    detection_report_path=DETECTION_REPORT,
    gt_directory=GT_DIRECTORY,
    logger=logger
)

# Generate and print report
summary = generate_update_report(results, UPDATE_REPORT)
print_update_summary(summary)

2026-01-24 08:16:45 - INFO - PMC7441611: Updated GT from 4 to 65 accessions (+61)
INFO:RangeExpander:PMC7441611: Updated GT from 4 to 65 accessions (+61)
2026-01-24 08:16:46 - INFO - PMC7557518: Updated GT from 17 to 45 accessions (+28)
INFO:RangeExpander:PMC7557518: Updated GT from 17 to 45 accessions (+28)
2026-01-24 08:16:46 - INFO - PMC7764154: Updated GT from 2 to 9 accessions (+7)
INFO:RangeExpander:PMC7764154: Updated GT from 2 to 9 accessions (+7)



GROUND TRUTH UPDATE SUMMARY
Update timestamp:            2026-01-24T08:16:46.936821
Files processed:             3
Successful updates:          3
Failed updates:              0
----------------------------------------------------------------------
Original accessions:         23
New accessions:              119
Accessions added:            96

